# Exploring the platform

A short tour of the research platform, using the same objects the pipeline
uses. Everything here reads the artefacts produced by
`python -m experiments.run_all`; nothing is recomputed from scratch, so this
notebook is fast and cannot disagree with the report.

Run the pipeline first if `data/processed/` is empty.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 30)

## 1. Load the cleaned dataset

`MarketData` is the single input contract for every stage. It carries the
adjusted and unadjusted price panels, the investability mask, and the record
of which values were forward filled.

In [ ]:
from src.data.loader import MarketData
from src.utils.config import load_config

cfg = load_config(ROOT / "config")
market = MarketData.from_processed(cfg.path("processed"), cfg.path("metadata"),
                                   cfg.asset_class_map, cfg.group_map)

print(f"data version  : {market.data_version}")
print(f"config hash   : {cfg.fingerprint()}")
print(f"universe      : {len(market.tickers)} assets")
print(f"calendar      : {len(market.index)} days, "
      f"{market.index.min().date()} to {market.index.max().date()}")
# `inception` is a timestamp column, so round only the numeric ones.
overview = market.describe()
overview.select_dtypes(include=[float]).round(4).join(overview[['inception']])

## 2. Why the corporate-action treatment matters

Total return minus price-only return is the distribution contribution. On
price returns alone, high-yield credit appears to lose money.

In [ ]:
comparison = pd.DataFrame({
    "total_return_ann": market.returns().mean() * 252,
    "price_only_ann": market.price_returns().mean() * 252,
    "distributions": market.dividend_yield(),
}).sort_values("distributions", ascending=False)
comparison.round(4)

## 3. How many independent bets does the universe contain?

PCA on the correlation matrix, not the covariance matrix -- otherwise PC1
simply rediscovers that silver is volatile.

In [ ]:
from src.features.pca import marchenko_pastur_bounds, pca_decomposition, significant_components

returns = market.returns().dropna(how="any")
pca = pca_decomposition(returns, use_correlation=True)

print(f"components for 90% of variance : {pca.n_components_for(0.90)}")
print(f"effective rank                 : {pca.effective_rank():.2f}")
print(f"above the noise bound (252d)   : {significant_components(pca, 252)}")
pca.summary(5).round(4)

## 4. The decisive question: does anything survive costs?

`breakeven_cost` is the transaction cost at which a strategy's net return is
exactly zero. It is the single most useful number for deciding whether a
signal is implementable.

In [ ]:
from src.backtest.engine import BacktestEngine
from src.features.volatility import rolling_volatility
from src.signals.transform import signal_to_positions
from src.features.momentum import volatility_scaled_momentum
from experiments.strategies import transform_config

signal = volatility_scaled_momentum(market.prices, market.returns(), 126, 1, 63)
weights = signal_to_positions(signal.where(market.investable),
                              rolling_volatility(market.returns(), 63),
                              investable=market.investable, **transform_config(cfg))

engine = BacktestEngine.from_config(cfg)
result = engine.run(weights, market.returns(), "momentum", market.investable,
                    apply_vol_target=True)

summary = result.summary()
print(f"gross Sharpe   : {summary['gross_sharpe']:+.3f}")
print(f"net Sharpe     : {summary['sharpe']:+.3f}")
print(f"turnover       : {summary['ann_turnover']:.1f}x per year")
print(f"breakeven cost : {result.breakeven_cost_bps():.1f} bps")
result.cost_sensitivity().round(4)

## 5. Prove the strategy cannot see the future

Scramble every observation after a split date and require every weight before
it to be bit-identical. A deliberately broken control strategy is included,
because a test that cannot fail proves nothing.

In [ ]:
from src.validation.leakage import check_no_lookahead

def honest(m):
    s = volatility_scaled_momentum(m.prices, m.returns(), 126, 1, 63).where(m.investable)
    return signal_to_positions(s, rolling_volatility(m.returns(), 63),
                               investable=m.investable, **transform_config(cfg))

def cheating(m):
    return signal_to_positions(m.returns().shift(-1), rolling_volatility(m.returns(), 63),
                               investable=m.investable, **transform_config(cfg))

for name, builder in [("honest momentum", honest), ("CONTROL (uses tomorrow)", cheating)]:
    outcome = check_no_lookahead(market, builder, "2017-06-30")
    print(f"{name:26s} -> {outcome.message}")

## 6. The generated results

Every table the report cites is a CSV in `reports/tables/`, and every figure
has a caption file stating the research question it answers.

In [ ]:
tables = sorted((ROOT / "reports" / "tables").glob("*.csv"))
print(f"{len(tables)} tables generated\n")

final = pd.read_csv(ROOT / "reports" / "tables" / "stage12_final_comparison.csv", index_col=0)
final[["cagr", "ann_vol", "sharpe", "max_drawdown", "ann_turnover"]].astype(float).round(4)

In [ ]:
holdout = pd.read_csv(ROOT / "reports" / "tables" / "stage12_final_holdout.csv", index_col=0)
print("Final holdout (2022 onwards), opened once after model selection was frozen:")
holdout[["cagr", "ann_vol", "sharpe", "max_drawdown"]].astype(float).round(4).sort_values("sharpe", ascending=False)

## 7. Every experiment, including the rejections

The registry is append-only and records the decision for each hypothesis. A
research process that reports only its successes is not a research
process.

In [ ]:
from src.utils.experiments import ExperimentRegistry

registry = ExperimentRegistry(ROOT / "experiments" / "registry.jsonl")
records = registry.records()
frame = pd.DataFrame([{
    "id": r["experiment_id"], "stage": r["stage"],
    "decision": r["decision"].upper(), "hypothesis": r["hypothesis"][:78],
} for r in records])
print(frame["decision"].value_counts().to_string(), "\n")
frame